# Gold: hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` (one row per hospital per day) from `silver.lab_results`, LEFT JOINed to hospitals and calendar, with LAG-based day-over-day trend. Full rebuild per STTM `sttm_silver_to_gold.xlsx`.

Note: `silver.hospitals` is deduplicated per `hospital_id` (latest `updated_timestamp`) before joining.

In [0]:
import re

dbutils.widgets.text("catalog", "adb_carsync", "Catalog")
catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: {catalog}")

target_table = f"{catalog}.gold.hospital_daily_summary"
print(f"Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH hospitals_dedup AS (
  SELECT hospital_id, hospital_name
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY updated_timestamp DESC) AS rn
    FROM {catalog}.silver.hospitals
  )
  WHERE rn = 1
),
daily AS (
  SELECT
    hospital_id,
    test_date                             AS summary_date,
    COUNT(*)                              AS total_tests,
    COUNT(DISTINCT patient_id)            AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)             AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM {catalog}.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT *,
         LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  d.hospital_id,
  h.hospital_name,
  d.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(d.total_tests AS INT)             AS total_tests,
  CAST(d.unique_patients_tested AS INT)  AS unique_patients_tested,
  CAST(d.active_doctor_count AS INT)     AS active_doctor_count,
  CAST(d.abnormal_test_count AS INT)     AS abnormal_test_count,
  CAST(d.critical_test_count AS INT)     AS critical_test_count,
  CAST(d.abnormal_test_count * 100.0 / d.total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(d.critical_test_count * 100.0 / d.total_tests AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(d.prev_day_total_tests AS INT)    AS prev_day_total_tests,
  CAST(
    CASE WHEN d.prev_day_total_tests IS NULL OR d.prev_day_total_tests = 0 THEN NULL
         ELSE (d.total_tests - d.prev_day_total_tests) * 100.0 / d.prev_day_total_tests
    END AS DECIMAL(6,2))                 AS test_volume_change_pct
FROM with_lag d
LEFT JOIN hospitals_dedup h
  ON d.hospital_id = h.hospital_id
LEFT JOIN {catalog}.silver.calendar c
  ON c.date_key = CAST(DATE_FORMAT(d.summary_date, 'yyyyMMdd') AS INT)
""")

In [0]:
tgt = spark.table(target_table)
tgt_cnt = tgt.count()
distinct_cnt = tgt.select("hospital_id", "summary_date").distinct().count()
src_tests = spark.table(f"{catalog}.silver.lab_results").count()
tgt_tests = tgt.agg({"total_tests": "sum"}).collect()[0][0]

print(f"gold rows: {tgt_cnt} | distinct (hospital_id, summary_date): {distinct_cnt}")
print(f"silver.lab_results rows: {src_tests} | sum(total_tests): {tgt_tests}")
assert tgt_cnt == distinct_cnt, "Grain violated: duplicate hospital/day"
assert tgt_tests == src_tests, "Test totals do not reconcile with silver.lab_results"

display(tgt.orderBy("hospital_id", "summary_date").limit(10))

In [0]:
updated_df = spark.table(target_table)

record_count = updated_df.count()
print("Processed Record Count:", record_count)

dbutils.notebook.exit(str(record_count))